In [1]:
import numpy as np
from glob import glob

for file_name in glob("data/*/*.npy"):
    content = np.load(file_name)
    print(file_name, content.shape)

data/FetchPickAndPlace/act_grasp_at_later.npy (5123, 50, 4)
data/FetchPickAndPlace/next_obs_grasp_at_5.npy (4685, 50, 28)
data/FetchPickAndPlace/obs_grasp_at_5.npy (4685, 50, 28)
data/FetchPickAndPlace/obs_grasp_at_later.npy (5123, 50, 28)
data/FetchPickAndPlace/next_obs_grasp_at_later.npy (5123, 50, 28)
data/FetchPickAndPlace/act_grasp_at_5.npy (4685, 50, 4)
data/BilliardBall/no_pocket_drop_1.npy (1200, 201, 2)
data/BilliardBall/pocket_drop_150.npy (1200, 201, 2)
data/BilliardBall/no_pocket_drop_2.npy (1200, 201, 2)
data/BilliardBall/no_pocket_drop_3.npy (1200, 201, 2)
data/BilliardBall/pocket_drop_50.npy (1200, 201, 2)
data/BilliardBall/pocket_drop_100.npy (1200, 201, 2)


In [6]:
from pathlib import Path
import torch
from torch.utils.data import Dataset
import numpy as np
from glob import glob


class TimeSeriesDatasetV1(Dataset):
    """
    PyTorch Dataset for time series data to train RNNs where the label is
    the delta between the current state and the next state.

    Args:
        time_series (numpy.ndarray): Time series data with shape (batch_size, seq_length, n_features)
        transform (callable, optional): Optional transform to be applied on the features
        target_transform (callable, optional): Optional transform to be applied on the targets
    """

    def __init__(
        self,
        time_series: np.ndarray,
        transform=None,
        target_transform=None,
    ):
        assert len(time_series.shape) >= 3, (
            "Expected time series data with shape (batch_size, seq_length, n_features)"
        )
        self.time_series = time_series
        self.transform = transform if transform is not None else torch.FloatTensor
        self.target_transform = (
            target_transform if target_transform is not None else torch.FloatTensor
        )

    def __len__(self):
        n_samples, _, _ = self.time_series.shape
        return n_samples

    def __getitem__(self, idx):
        # Extract the sequence
        sequence = self.time_series[idx, :-1]

        # Get the next state
        next_state = self.time_series[idx, 1:]

        # Calculate delta between the current (last state in sequence) and next state
        delta = next_state - sequence

        # Apply transformations if specified
        sequence = self.transform(sequence)
        delta = self.target_transform(delta)

        return sequence, delta

    @classmethod
    def from_file(
        cls, file_name: Path | str, transform=None, target_transform=None
    ) -> "TimeSeriesDatasetV1":
        time_series = np.load(file_name)
        return cls(time_series, transform, target_transform)


class BilliardBall(TimeSeriesDatasetV1):
    def __init__(self, time_series, transform=None, target_transform=None):
        super().__init__(time_series, transform, target_transform)


class FetchPickAndPlace(TimeSeriesDatasetV1):
    def __init__(
        self,
        states: np.ndarray,
        actions: np.ndarray,
        next_states: np.ndarray,
        transform=None,
        target_transform=None,
    ):
        self.state_dim = states.shape[-1]
        self.action_dim = actions.shape[-1]
        time_series = np.concat([states, actions, next_states], axis=-1)
        super().__init__(time_series, transform, target_transform)

    def __getitem__(self, idx):
        ts = self.time_series[idx]

        x = ts[:, : self.state_dim + self.action_dim]
        current_state = x[:, : self.state_dim]
        next_state = ts[:, -self.state_dim:]

        delta = next_state - current_state

        # Apply transformations if specified
        x = self.transform(x)
        delta = self.target_transform(delta)

        return x, delta

    @classmethod
    def from_file(cls, file_name, transform=None, target_transform=None):
        raise NotImplementedError


class TimeSeriesDatasetV2(Dataset):
    """
    PyTorch Dataset for time series data to train RNNs where the label is
    the delta between the current state and the next state.

    Args:
        time_series (numpy.ndarray): Time series data with shape (batch_size, seq_length, n_features)
        context_length (int): Number of time steps to use for each sequence
        transform (callable, optional): Optional transform to be applied on the features
        target_transform (callable, optional): Optional transform to be applied on the targets
    """

    def __init__(
        self,
        time_series: np.ndarray,
        context_length: int = None,
        transform=None,
        target_transform=None,
    ):
        assert len(time_series.shape) >= 3, (
            "Expected time series data with shape (batch_size, seq_length, n_features)"
        )
        self.time_series = time_series
        self.context_length = (
            context_length
            if context_length is not None
            else self.time_series.shape[1] - 1
        )
        self.transform = transform if transform is not None else torch.FloatTensor
        self.target_transform = (
            target_transform if target_transform is not None else torch.FloatTensor
        )

    def __len__(self):
        n_samples, seq_length, _ = self.time_series.shape
        return n_samples * (seq_length - self.context_length)

    def __getitem__(self, idx):
        # Get the starting position for this sequence
        ts_idx = int(idx / (self.time_series.shape[1] - self.context_length))
        t_idx = idx - (self.time_series.shape[1] - self.context_length) * ts_idx

        # Extract the sequence
        sequence = self.time_series[ts_idx, t_idx : t_idx + self.context_length]

        # Get the next state
        next_state = self.time_series[ts_idx, t_idx + self.context_length]

        # Calculate delta between the current (last state in sequence) and next state
        delta = next_state - sequence[-1]

        # Apply transformations if specified
        sequence = self.transform(sequence)
        delta = self.target_transform(delta)

        return sequence, delta

    @classmethod
    def from_file(
        cls,
        file_name: Path | str,
        context_length=None,
        transform=None,
        target_transform=None,
    ) -> "TimeSeriesDatasetV2":
        time_series = np.load(file_name)
        return cls(time_series, context_length, transform, target_transform)


def load_billard_ball(normalize: bool = False) -> BilliardBall:
    content = []
    for file_name in glob("data/BilliardBall/*.npy"):
        content.append(np.load(file_name))
    content = np.concat(content, axis=0)

    if normalize:
        content -= content.mean()
        content /= content.std()

    return BilliardBall(content, transform=None, target_transform=None)


def load_fetch_pick_and_place(normalize: bool = False) -> FetchPickAndPlace:
    current_state = np.concat(
        [np.load(file_name) for file_name in glob("data/FetchPickAndPlace/obs_*.npy")],
        axis=0,
    )
    actions = np.concat(
        [np.load(file_name) for file_name in glob("data/FetchPickAndPlace/act_*.npy")],
        axis=0,
    )
    next_state = np.concat(
        [np.load(file_name) for file_name in glob("data/FetchPickAndPlace/next_obs_*.npy")],
        axis=0,
    )

    if normalize:
        current_state -= current_state.mean()
        current_state /= current_state.std()
        actions -= actions.mean()
        actions /= actions.std()
        next_state -= next_state.mean()
        next_state /= next_state.std()

    return FetchPickAndPlace(current_state, actions, next_state, transform=None, target_transform=None)


len(load_billard_ball())

7200

In [7]:
ds = load_fetch_pick_and_place()
x, y = ds[0]
x.shape, y.shape

(torch.Size([50, 32]), torch.Size([50, 28]))

In [24]:
from project.gatel0rd import GateL0RDv0


rnn = GateL0RDv0(2, 8, 2, 3, 3, 3)
len(torch.cat(list(map(lambda x: x.flatten(), rnn.parameters()))))

732

In [ ]:
from typing import List
from torch import nn
from project import gatel0rd


def build_mlp(architecture: List[int], activation_function: str):
    assert len(architecture) >= 2, "At least input and output dimension are needed"

    layers = []
    for idx, dimension in enumerate(architecture[:-1]):
        layers.append(nn.Linear(dimension, architecture[idx + 1]))
        layers.append(getattr(nn, activation_function))
    layers = layers[:-1]  # cut of last activation function
    net = nn.Sequential(*layers)
    
    return net


class RNN(nn.Module):
    def __init__(
        self,
        gatelord_version: int,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        init_arc: List[int],
        pre_arc: List[int],
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.pre_net: 
        self.init_net: nn.Module

        self.rnn = getattr(gatel0rd, "GateL0RDv" + str(gatelord_version))
        self.rnn = self.rnn()

In [ ]:
from torch.utils.data import DataLoader

ds = load_fetch_pick_and_place()
rnn = GateL0RDv0(32, 8, 28, 3, 3, 3)

ds = load_billard_ball()
rnn = GateL0RDv0(2, 8, 2, 3, 3, 3)

dl = DataLoader(ds, 128)
    
for x, y in dl:
    rnn.forward(x)

In [ ]:
from torch import nn

def create_f_pre(f_pre_layers: int, input_dim: int, feature_dim: int) -> nn.Sequential:
    """create preprocessing subnet. Fan in type of network, decreasing features per layer

    Args:
        f_pre_layers (int): number of layers
        input_dim (int): amount of input features
        feature_dim (int): amount of output neurons

    Returns:
        nn.Sequential: preprocessing network
    """
    module_pre = nn.ModuleList([])
    h_dim = input_dim
    for pre_l in range(f_pre_layers):
        # Fan in type of network, decreasing features per layer
        pre_l_factor = pow(2, (f_pre_layers - pre_l - 1))
        module_pre.append(nn.Linear(h_dim, pre_l_factor * feature_dim))
        module_pre.append(nn.Tanh())
        h_dim = pre_l_factor * feature_dim
    return nn.Sequential(*module_pre)

def create_f_init(f_init_layers: int, f_init_inputs: int, input_dim: int, feature_dim: int, latent_dim: int) -> nn.Sequential:
    """create network to initialize the hidden state

    Args:
        f_init_layers (int): amount of layers
        f_init_inputs (int): 
        input_dim (int): _description_
        feature_dim (int): _description_
        latent_dim (int): _description_

    Returns:
        nn.Sequential: network to init hidden state
    """
    input_dim_warm_up = input_dim * f_init_inputs
    feature_dim_warm_up = feature_dim
    warm_up_net = nn.ModuleList([])
    for w in range(f_init_layers):
        w_factor = pow(2, (f_init_layers - w - 1))
        if w == (f_init_layers - 1):
            feature_dim_warm_up = latent_dim
        warm_up_net.append(nn.Linear(input_dim_warm_up, w_factor * feature_dim_warm_up))
        warm_up_net.append(nn.Tanh())
        input_dim_warm_up = w_factor * feature_dim
    return nn.Sequential(*warm_up_net)



def create_f_post(f_post_layers: int, feature_dim: int, output_dim: int) -> nn.Sequential:
    """create output network

    Args:
        f_post_layers (int): _description_
        feature_dim (int): _description_
        output_dim (int): _description_

    Returns:
        nn.Sequential: network to map output into desired space
    """
    post_module = nn.ModuleList([])
    in_post = feature_dim
    h_dim = feature_dim
    for post_l in range(f_post_layers):
        h_factor = pow(2, (f_post_layers - post_l - 2))
        if post_l == f_post_layers - 1:
            h_factor = 1
            h_dim = output_dim
        post_module.append(nn.Linear(in_post, h_factor * h_dim))
        if post_l < (f_post_layers - 1):
            post_module.append(nn.Tanh())
        in_post = h_factor * h_dim
    return nn.Sequential(*post_module)


create_f_post(3, 16, 2), create_f_pre(3, 16, 2)     

(Sequential(
   (0): Linear(in_features=16, out_features=32, bias=True)
   (1): Tanh()
   (2): Linear(in_features=32, out_features=16, bias=True)
   (3): Tanh()
   (4): Linear(in_features=16, out_features=2, bias=True)
 ),
 Sequential(
   (0): Linear(in_features=16, out_features=8, bias=True)
   (1): Tanh()
   (2): Linear(in_features=8, out_features=4, bias=True)
   (3): Tanh()
   (4): Linear(in_features=4, out_features=2, bias=True)
   (5): Tanh()
 ))

a



[None, None, None, None, None]

torch.Size([1, 4, 6])